# 01 — Exploração dos microdados de vacinação (SI-PNI)

**Objetivo:** entender o arquivo bruto do OpenDataSUS antes de escrever o ETL.
Este notebook **não é código de produção**: é o mapa para os próximos cards.

- **Arquivo:** `etl/data/raw/vacinacao_jan_2024.csv` (microdados nacionais de jan/2024, ~6,9 GB)
- **Fonte:** OpenDataSUS — Vacinação (registros individuais anonimizados)
- **Recorte do MVP:** Pernambuco (PE), 2024
- **Indicador alvo:** doses aplicadas por município × vacina × mês

Roteiro:
1. Olhar o cabeçalho e uma amostra de **todas** as colunas (tipos, exemplos, % de nulos).
2. Ler o arquivo inteiro **em chunks**, só com as colunas necessárias, filtrando PE em cada chunk.
3. Analisar o recorte de PE: datas, código de município, vacinas, nulos, duplicados.
4. Anotar as descobertas e a lista de colunas que o ETL vai usar.

In [1]:
import time
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)

# O notebook roda a partir de etl/notebooks/
ETL_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARQUIVO_CSV = ETL_DIR / "data" / "raw" / "vacinacao_jan_2024.csv"
CACHE_PE = ETL_DIR / "data" / "processed" / "vacinacao_pe_2024_01.parquet"

# Formato do arquivo (descoberto com `file` e `head`): ver seção 1
CSV_OPCOES = {
    "sep": ";",
    "encoding": "latin-1",
    "dtype": str,  # tudo como texto: preserva zeros à esquerda (ex.: raça '01', CEP '09412')
    "keep_default_na": False,
    "na_values": [""],  # só campo vazio é nulo ('NA' poderia ser um valor legítimo)
}

print(ARQUIVO_CSV, f"{ARQUIVO_CSV.stat().st_size / 1024**3:.2f} GB")

/home/swetony/Documentos/meus-projetos/painel-saude-aberta/etl/data/raw/vacinacao_jan_2024.csv 6.80 GB


## 1. Formato do arquivo

Inspeção feita no terminal (`file`, `head`):

| Item | Valor |
|---|---|
| Separador | `;` |
| Codificação | **ISO-8859-1 (latin-1)** — ler como UTF-8 corrompe acentos (`Dose` com `ª`, `Prontuário`) |
| Quebra de linha | CRLF (Windows) — o pandas trata sozinho |
| Cabeçalho | sim, nomes em `snake_case` minúsculo |
| Colunas | 56 |
| Aspas | não há campos entre aspas |

In [2]:
amostra = pd.read_csv(ARQUIVO_CSV, nrows=200_000, **CSV_OPCOES)
print(
    f"{amostra.shape[1]} colunas, amostra de {len(amostra):,} linhas (primeiras do arquivo, todas as UFs)"
)

perfil = pd.DataFrame(
    {
        "exemplo": amostra.apply(
            lambda c: c.dropna().iloc[0] if c.notna().any() else None
        ),
        "pct_nulo": (amostra.isna().mean() * 100).round(1),
        "n_distintos": amostra.nunique(),
    }
)
perfil

56 colunas, amostra de 200,000 linhas (primeiras do arquivo, todas as UFs)


,exemplo,pct_nulo,n_distintos
co_documento,821d31ff-f739-4e0f-a080-e564ce18e569-i0b0,0.0,200000
co_paciente,0ecf19489ac4a44e4e9294d4a704bb0b1c30c28bd0608ecf3fddc26278cae7cf,0.0,197369
tp_sexo_paciente,F,0.0,3
co_raca_cor_paciente,01,0.0,6
no_raca_cor_paciente,BRANCA,0.0,6
co_municipio_paciente,261220,1.6,5411
co_pais_paciente,10,1.6,4
no_municipio_paciente,SALGUEIRO,1.6,5148
no_pais_paciente,BRASIL,1.6,4
sg_uf_paciente,PE,1.6,28


### Tipos
Tudo foi lido como texto (`str`) de propósito. Os tipos **reais** que o ETL deve aplicar:

| Tipo | Colunas (exemplos) |
|---|---|
| texto com zeros à esquerda (manter `str`) | `co_municipio_*`, `co_raca_cor_paciente`, `nu_cep_paciente`, `co_vacina_grupo_atendimento` |
| data `YYYY-MM-DD` | `dt_vacina`, `dt_entrada_rnds`, `dt_deletado_rnds` |
| inteiro | `nu_idade_paciente`, `co_vacina`, `co_dose_vacina` |
| categórico (código + descrição) | `co_vacina`/`sg_imunobiologico`/`ds_nome`, `co_dose_vacina`/`ds_tipo_dose`, `co_estrategia_vacinacao`/`no_estrategia` |
| identificador pseudonimizado | `co_documento` (registro), `co_paciente` (hash da pessoa) |

A célula abaixo confirma o que o pandas inferiria se deixássemos a inferência automática.

In [3]:
inferido = pd.read_csv(ARQUIVO_CSV, nrows=50_000, sep=";", encoding="latin-1")
inferido.dtypes.value_counts(), inferido.dtypes[inferido.dtypes != "object"]

/tmp/ipykernel_49489/102132752.py:1: DtypeWarning: Columns (13: co_etnia_indigena_paciente) have mixed types. Specify dtype option on import or set low_memory=False.
  inferido = pd.read_csv(ARQUIVO_CSV, nrows=50_000, sep=";", encoding="latin-1")


(str        35
 float64    13
 int64       7
 object      1
 Name: count, dtype: int64,
 co_documento                           str
 co_paciente                            str
 tp_sexo_paciente                       str
 co_raca_cor_paciente                 int64
 no_raca_cor_paciente                   str
 co_municipio_paciente              float64
 co_pais_paciente                   float64
 no_municipio_paciente                  str
 no_pais_paciente                       str
 sg_uf_paciente                         str
 nu_cep_paciente                    float64
 ds_nacionalidade_paciente              str
 no_etnia_indigena_paciente             str
 co_cnes_estabelecimento              int64
 no_razao_social_estabelecimento        str
 no_fantasia_estalecimento              str
 co_municipio_estabelecimento         int64
 no_municipio_estabelecimento           str
 sg_uf_estabelecimento                  str
 co_troca_documento                     str
 co_vacina                      

> ⚠️ Com inferência automática, códigos como `co_municipio_paciente` viram `float64`
> (por causa dos nulos) e `co_raca_cor_paciente` vira `int` (perde o zero à esquerda).
> **Por isso o ETL vai ler com `dtype=str` e converter explicitamente só o que precisa.**

## 2. Leitura em chunks filtrando PE

O arquivo nacional não cabe confortavelmente em memória. Estratégia:

- `usecols`: só as colunas que interessam ao indicador e às checagens de qualidade
  (**não** lemos `co_paciente` nem dados pessoais — não precisamos deles);
- `chunksize`: lê 500 mil linhas por vez;
- filtra PE **dentro de cada chunk**, antes de acumular;
- guarda o recorte de PE em parquet (`data/processed/`) para não reler os 6,9 GB a cada execução.

Lemos linhas onde **o estabelecimento ou o paciente** é de PE, para comparar os dois recortes na seção 3.

In [4]:
COLUNAS = [
    "co_documento",  # id do registro -> checagem de duplicados
    "co_municipio_estabelecimento",
    "no_municipio_estabelecimento",
    "sg_uf_estabelecimento",
    "co_municipio_paciente",
    "sg_uf_paciente",
    "co_vacina",
    "sg_imunobiologico",
    "ds_nome",
    "dt_vacina",
    "co_dose_vacina",
    "ds_tipo_dose",
    "st_documento",
    "co_troca_documento",
    "dt_deletado_rnds",
]
UF = "PE"


def ler_pe_em_chunks(
    caminho: Path, chunksize: int = 500_000
) -> tuple[pd.DataFrame, int]:
    """Lê o CSV nacional em pedaços e devolve só as linhas de PE (estabelecimento ou paciente)."""
    partes: list[pd.DataFrame] = []
    linhas_lidas = 0
    inicio = time.perf_counter()
    for i, chunk in enumerate(
        pd.read_csv(caminho, usecols=COLUNAS, chunksize=chunksize, **CSV_OPCOES)
    ):
        linhas_lidas += len(chunk)
        filtro = (chunk["sg_uf_estabelecimento"] == UF) | (
            chunk["sg_uf_paciente"] == UF
        )
        partes.append(chunk.loc[filtro])
        if i % 4 == 0:
            print(
                f"chunk={i} linhas_lidas={linhas_lidas:,} tempo={time.perf_counter() - inicio:.0f}s"
            )
    print(
        f"fim linhas_lidas={linhas_lidas:,} tempo={time.perf_counter() - inicio:.0f}s"
    )
    return pd.concat(partes, ignore_index=True), linhas_lidas


if CACHE_PE.exists():
    pe = pd.read_parquet(CACHE_PE)
    print(f"cache carregado: {CACHE_PE.name}")
else:
    pe, total_nacional = ler_pe_em_chunks(ARQUIVO_CSV)
    pe.attrs["total_nacional"] = total_nacional
    CACHE_PE.parent.mkdir(parents=True, exist_ok=True)
    pe.to_parquet(CACHE_PE, index=False)

print(f"linhas com PE (estab. ou paciente): {len(pe):,}")
print(f"memória do recorte: {pe.memory_usage(deep=True).sum() / 1024**2:.0f} MB")

chunk=0 linhas_lidas=500,000 tempo=2s


chunk=4 linhas_lidas=2,500,000 tempo=12s


chunk=8 linhas_lidas=4,500,000 tempo=24s


chunk=12 linhas_lidas=6,500,000 tempo=34s


chunk=16 linhas_lidas=8,500,000 tempo=45s


chunk=20 linhas_lidas=10,500,000 tempo=55s


chunk=24 linhas_lidas=12,446,851 tempo=66s
fim linhas_lidas=12,446,851 tempo=66s


linhas com PE (estab. ou paciente): 575,888
memória do recorte: 145 MB


## 3. Análise do recorte de PE

### 3.1 Qual UF usar: do estabelecimento ou do paciente?

O indicador é **doses aplicadas** no município. Comparamos os dois recortes:

In [5]:
estab_pe = pe["sg_uf_estabelecimento"] == UF
pac_pe = pe["sg_uf_paciente"] == UF
pd.Series(
    {
        "estabelecimento PE": estab_pe.sum(),
        "paciente PE": pac_pe.sum(),
        "ambos PE": (estab_pe & pac_pe).sum(),
        "aplicada em PE, paciente de fora (ou sem UF)": (estab_pe & ~pac_pe).sum(),
        "paciente de PE, aplicada fora": (~estab_pe & pac_pe).sum(),
    },
    name="linhas",
).to_frame()

,linhas
estabelecimento PE,555787
paciente PE,541557
ambos PE,521456
"aplicada em PE, paciente de fora (ou sem UF)",34331
"paciente de PE, aplicada fora",20101


In [6]:
# Daqui em diante: só doses aplicadas em estabelecimentos de PE
doses = pe.loc[estab_pe].copy()
print(f"{len(doses):,} doses aplicadas em PE em jan/2024")

555,787 doses aplicadas em PE em jan/2024


### 3.2 % de nulos nas colunas selecionadas

In [7]:
(doses.isna().mean() * 100).round(2).sort_values(ascending=False).to_frame("pct_nulo")

,pct_nulo
dt_deletado_rnds,100.00
co_troca_documento,95.14
sg_uf_paciente,2.58
co_municipio_paciente,2.58
co_documento,0.00
no_municipio_estabelecimento,0.00
co_municipio_estabelecimento,0.00
sg_uf_estabelecimento,0.00
co_vacina,0.00
dt_vacina,0.00


### 3.3 Formato da data (`dt_vacina`)

In [8]:
print("tamanhos:", doses["dt_vacina"].str.len().value_counts().to_dict())
print(
    "bate com YYYY-MM-DD:", doses["dt_vacina"].str.fullmatch(r"\d{4}-\d{2}-\d{2}").all()
)
datas = pd.to_datetime(doses["dt_vacina"], format="%Y-%m-%d", errors="coerce")
print("inválidas:", datas.isna().sum())
print("mín:", datas.min().date(), "| máx:", datas.max().date())
datas.dt.to_period("M").value_counts().sort_index()

tamanhos: {10: 555787}
bate com YYYY-MM-DD: True


inválidas: 0
mín: 2024-01-01 | máx: 2024-01-31


dt_vacina
2024-01    555787
Freq: M, Name: count, dtype: int64

### 3.4 Formato do código de município (6 ou 7 dígitos?)

In [9]:
cod = doses["co_municipio_estabelecimento"]
print("tamanhos:", cod.str.len().value_counts().to_dict())
print("só dígitos:", cod.str.fullmatch(r"\d+").all())
print("prefixo UF (26 = PE):", cod.str[:2].value_counts().to_dict())
print(
    "municípios distintos:",
    cod.nunique(),
    "(PE tem 185 municípios, incluindo Fernando de Noronha)",
)
# um código -> um nome?
nomes_por_cod = doses.groupby("co_municipio_estabelecimento")[
    "no_municipio_estabelecimento"
].nunique()
print("códigos com mais de um nome:", (nomes_por_cod > 1).sum())
doses[
    ["co_municipio_estabelecimento", "no_municipio_estabelecimento"]
].drop_duplicates().head(5)

tamanhos: {6: 555787}
só dígitos: True
prefixo UF (26 = PE): {'26': 555787}
municípios distintos: 185 (PE tem 185 municípios, incluindo Fernando de Noronha)
códigos com mais de um nome: 1


,co_municipio_estabelecimento,no_municipio_estabelecimento
0,261220,SALGUEIRO
1,261160,RECIFE
2,260750,ITAIBA
3,261070,PAULISTA
5,260390,CARNAIBA


In [10]:
# Qual código tem mais de um nome?
multi = nomes_por_cod[nomes_por_cod > 1].index
doses.loc[doses["co_municipio_estabelecimento"].isin(multi)].groupby(
    ["co_municipio_estabelecimento", "no_municipio_estabelecimento"]
).size()

co_municipio_estabelecimento  no_municipio_estabelecimento
260160                        BELEM DE SAO FRANCISCO          994
                              BELEM DO SAO FRANCISCO            9
dtype: int64

### 3.5 Vacinas distintas

In [11]:
vacinas = (
    doses.groupby(["co_vacina", "sg_imunobiologico", "ds_nome"], dropna=False)
    .size()
    .rename("doses")
    .reset_index()
    .sort_values("doses", ascending=False)
)
print("co_vacina distintos:", doses["co_vacina"].nunique())
print("sg_imunobiologico distintos:", doses["sg_imunobiologico"].nunique())
print("ds_nome distintos:", doses["ds_nome"].nunique())
print("combinações (co, sigla, nome):", len(vacinas))
vacinas

co_vacina distintos: 75
sg_imunobiologico distintos: 75
ds_nome distintos: 61
combinações (co, sigla, nome): 75


,co_vacina,sg_imunobiologico,ds_nome,doses
5,103,COVID-19 PFIZER - BI,vacina covid-19 bivalente,78817
70,9,HB,vacina hepatite B (recombinante),38736
12,14,VFA,vacina febre amarela (atenuada),37148
35,42,Penta,"vacina adsorvida difteria, tétano, pertussis, hepatite B (recombin...",35836
23,25,dT,vacina adsorvida difteria e tétano adulto,35060
20,22,VIP,"vacina poliomelite 1, 2 e 3 (inativada)",34969
24,26,VPC10,vacina pneumocócica 10-valente (conjugada),33527
57,74,MenACWY,vacina meningocócica ACWY (conjugada),26147
22,24,SCR,"vacina sarampo, caxumba, rubéola (atenuada)",25321
34,41,MenC,vacina meningocócica C (conjugada),24125


In [12]:
# co_vacina -> sigla é 1:1? e sigla -> ds_nome?
print(
    "co_vacina com +1 sigla:",
    (doses.groupby("co_vacina")["sg_imunobiologico"].nunique() > 1).sum(),
)
print(
    "ds_nome com +1 co_vacina:",
    (doses.groupby("ds_nome")["co_vacina"].nunique() > 1).sum(),
)
doses.groupby("ds_nome")["sg_imunobiologico"].unique().loc[
    lambda s: s.str.len() > 1
].to_frame("siglas agrupadas no mesmo ds_nome")

co_vacina com +1 sigla: 0


ds_nome com +1 co_vacina: 10


,siglas agrupadas no mesmo ds_nome
ds_nome,
vacina adsorvida covid-19 (inativada),"[COVID-19 SINOVAC, COV-19 SINO/BUTAN-CO]"
"vacina adsorvida difteria, tétano e pertussis (acelular)","[dTpa, DTPa]"
vacina adsorvida hepatite A (inativada),"[HAinf, HAad]"
vacina adsorvida hepatite A e hepatite B (recombinante),"[HA-HBad, HA-HBinf]"
vacina covid-19,"[COV19-RNAm-PED, COVID-19 PFIZER-<5a, COV-19 PFIZER-COMIRN, COVID-..."
vacina covid-19 (recombinante),"[COV19-Ad26, COVID19 SERUM/ZALIKA, COV-19 AST/FIO-CVISH, COV-19 AS..."
vacina febre amarela (atenuada),"[VFA, VFA-F]"
"vacina influenza tetravalente (fragmentada, inativada)","[INF4, INF4-alta dosagem]"
"vacina influenza tipo A (H1N1) (inativada, fragmentada)","[INFH1N1-2009, INFH1N1-2009-2]"


### 3.6 Tipo de dose

In [13]:
doses.groupby(["co_dose_vacina", "ds_tipo_dose"]).size().sort_values(
    ascending=False
).head(20)

co_dose_vacina  ds_tipo_dose                        
1               1ª Dose                                 160349
38              Reforço                                 118000
2               2ª Dose                                 108506
3               3ª Dose                                  46436
9               Única                                    35999
6               1º Reforço                               33692
8               Dose                                     25668
7               2º Reforço                               22439
32              1ª Dose Revacinação                       1436
33              2ª Dose Revacinação                        979
4               4ª Dose                                    656
10              Revacinação                                628
59              Profilaxia/Tratamento com 1 unidade        225
34              3ª Dose Revacinação                        139
36              Dose Inicial                               112
11

### 3.7 Status, exclusões, trocas e duplicados

In [14]:
print("st_documento:", doses["st_documento"].value_counts(dropna=False).to_dict())
print("dt_deletado_rnds preenchido:", doses["dt_deletado_rnds"].notna().sum())
print("co_troca_documento preenchido:", doses["co_troca_documento"].notna().sum())
print("co_documento duplicado:", doses["co_documento"].duplicated().sum())
# um registro de troca aponta para outro documento que também está no arquivo?
trocados = doses["co_troca_documento"].dropna()
print(
    "trocas cujo documento original também está no recorte:",
    trocados.isin(doses["co_documento"]).sum(),
)

st_documento: {'final': 555787}
dt_deletado_rnds preenchido: 0
co_troca_documento preenchido: 26997


co_documento duplicado: 0


trocas cujo documento original também está no recorte: 0


### 3.8 Prévia do indicador: município × vacina × mês

Esboço do que o `transform` vai produzir (é isso — e só isso — que vai para o banco).

In [15]:
doses["ano_mes"] = datas.dt.to_period("M").astype(str)
fato = (
    doses.groupby(["co_municipio_estabelecimento", "co_vacina", "ano_mes"])
    .size()
    .rename("total_doses")
    .reset_index()
)
print(f"{len(doses):,} microdados -> {len(fato):,} linhas agregadas")
print("total de doses confere:", fato["total_doses"].sum() == len(doses))
print(
    "chave única:",
    not fato.duplicated(["co_municipio_estabelecimento", "co_vacina", "ano_mes"]).any(),
)
fato.sort_values("total_doses", ascending=False).head(10)

555,787 microdados -> 4,725 linhas agregadas
total de doses confere: True
chave única: True


,co_municipio_estabelecimento,co_vacina,ano_mes,total_doses
3337,261160,103,2024-01,23877
3389,261160,9,2024-01,9767
3357,261160,33,2024-01,9175
3352,261160,25,2024-01,7129
3343,261160,14,2024-01,6840
3362,261160,42,2024-01,5224
2189,260790,103,2024-01,5171
3351,261160,24,2024-01,5125
3349,261160,22,2024-01,5055
3380,261160,74,2024-01,4799


In [16]:
# Municípios com mais doses (sanidade: Recife deve liderar)
doses.groupby(
    ["co_municipio_estabelecimento", "no_municipio_estabelecimento"]
).size().sort_values(ascending=False).head(10)

co_municipio_estabelecimento  no_municipio_estabelecimento
261160                        RECIFE                          119723
260790                        JABOATAO DOS GUARARAPES          33658
260410                        CARUARU                          21971
261110                        PETROLINA                        21480
261070                        PAULISTA                         15545
260960                        OLINDA                           15512
260290                        CABO DE SANTO AGOSTINHO           9706
261640                        VITORIA DE SANTO ANTAO            9599
260600                        GARANHUNS                         8814
260345                        CAMARAGIBE                        8776
dtype: int64

## 4. Descobertas

### 4.1 Arquivo
- `;` como separador, **latin-1**, CRLF, 56 colunas, cabeçalho em `snake_case`.
- O arquivo de jan/2024 tem **6,9 GB e 12.446.851 linhas** (bem acima dos ~1,4 GB estimados no planejamento).
  Ler em chunks de 500 mil linhas com 15 colunas levou ~80 s e o recorte de PE ocupa ~145 MB em memória.
- Inferência automática de tipos **estraga códigos** (`float64` por causa de nulos, perda de zero à esquerda)
  → ler tudo como `str` e converter explicitamente.

### 4.2 Recorte de PE
- **555.787 doses aplicadas em estabelecimentos de PE** em jan/2024 (~4,5% do país).
- UF do paciente ≠ UF do estabelecimento em ~6% dos casos (34 mil aplicadas em PE para pacientes de fora/sem UF;
  20 mil pacientes de PE vacinados fora). `co_municipio_paciente` tem 2,6% de nulos; o do estabelecimento, 0%.
- **Decisão:** o indicador "doses aplicadas por município" usa o **município do estabelecimento**
  (`co_municipio_estabelecimento` / `sg_uf_estabelecimento`). É onde a dose foi aplicada e não tem nulos.
  *(Cobertura vacinal por residência seria outro indicador — fora do MVP. Registrar no ADR-001.)*

### 4.3 Data
- `dt_vacina` sempre no formato **`YYYY-MM-DD`** (ISO), 0 inválidas, todas entre 2024-01-01 e 2024-01-31.
  → `pd.to_datetime(format="%Y-%m-%d")`; `ano_mes` = primeiro dia do mês.

### 4.4 Código de município
- **6 dígitos** (padrão DATASUS, sem dígito verificador), 100% numérico, prefixo `26` = PE.
- **185 municípios distintos** = todos os municípios de PE.
- O nome no microdado não é confiável (`260160` aparece como "BELEM **DE**" e "BELEM **DO** SAO FRANCISCO").
  → usar só o código; nome e código de 7 dígitos vêm da **tabela do IBGE** (dimensão `municipio`).
  Sugestão para o ADR: guardar os dois (`codigo_ibge7` como chave natural e `codigo_ibge6` para o join com o DATASUS).

### 4.5 Vacinas
- **75 `co_vacina` distintos** em PE, com relação **1:1 com `sg_imunobiologico`**.
- `ds_nome` é mais genérico (61 valores): agrupa siglas diferentes — ex.: 4+ produtos COVID-19 viram
  "vacina covid-19", `VFA`/`VFA-F` viram "vacina febre amarela".
- → **chave da dimensão `vacina` = `co_vacina`**, com `sg_imunobiologico` e `ds_nome` como atributos.
  Agrupar por `ds_nome` (ou por "família" de vacina) fica como opção de filtro no dashboard.
- Mais aplicadas: COVID-19 bivalente (78,8 mil), Hepatite B, Febre Amarela, Penta, dT.

### 4.6 Qualidade
| Checagem | Resultado em PE |
|---|---|
| Nulos em município / vacina / data | **0%** |
| `co_documento` duplicado | **0** |
| `st_documento` | 100% `final` |
| `dt_deletado_rnds` preenchido | **0** (100% nulo neste arquivo) |
| `co_troca_documento` preenchido | 4,9% — aponta para documentos que **não estão** no arquivo (correções de registros antigos) |
| Datas fora do mês | 0 |

- Ainda assim o ETL deve **filtrar** `st_documento == "final"` e `dt_deletado_rnds` nulo: hoje não descartam nada,
  mas protegem contra meses em que o arquivo traga registros excluídos.
- `co_troca_documento`: tratar como dose normal no MVP (cada linha = 1 dose aplicada); investigar se outros meses
  trazem o documento original junto (risco de contagem dupla) — vale virar uma checagem de qualidade.

### 4.7 Agregação
- 555.787 microdados → **4.725 linhas** em município × vacina × mês (redução de ~118×), chave única sem colisões.
- Recife lidera (119,7 mil doses), seguido de Jaboatão, Caruaru e Petrolina — bate com a população.
- Faixas para a checagem de "contagem dentro do intervalo esperado": ~500–600 mil linhas de entrada e ~4–5 mil
  linhas agregadas por mês em PE (calibrar com mais meses).

## 5. Colunas que o ETL vai usar

| Coluna | Uso | Tipo no ETL |
|---|---|---|
| `sg_uf_estabelecimento` | filtro da UF (o mais cedo possível, em cada chunk) | `str` |
| `co_municipio_estabelecimento` | dimensão município (join com IBGE 6 dígitos) | `str` (6 dígitos) |
| `co_vacina` | dimensão vacina (chave) | `str` → `int` |
| `sg_imunobiologico` | atributo da vacina (sigla) | `str` |
| `ds_nome` | atributo da vacina (nome) | `str` |
| `dt_vacina` | mês da dose (`ano_mes`) e checagem de período | `date` |
| `co_documento` | checagem de duplicados (não vai para o banco) | `str` |
| `st_documento` | filtro: só `final` | `str` |
| `dt_deletado_rnds` | filtro: descartar excluídos | `str`/`date` |

**Não usar:** `co_paciente` e demais dados do paciente (privacidade; desnecessários para o indicador),
`no_municipio_estabelecimento` (nome inconsistente — vem do IBGE), `co_municipio_paciente` (fora do MVP).

**Candidatas para depois:** `co_dose_vacina`/`ds_tipo_dose` (1ª dose, reforço…), `co_estrategia_vacinacao`
(rotina × campanha), `nu_idade_paciente` (faixa etária) — cada uma exige mudar o modelo, logo um ADR.

## 6. Próximos passos
1. ADR-001: agregação vs. microdados; município do estabelecimento; código IBGE 6 × 7.
2. Carregar a tabela de municípios do IBGE (dimensão).
3. `transform` com funções puras + testes usando uma fixture pequena em `etl/tests/fixtures/`.